<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2013%20-%20Decision%20Trees%20II/lab6_Decision_Trees_Loan_Acceptance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 6: Decision Trees (CART) - Universalbank revisited**

![source: https://pixels.com/featured/colorful-abstract-trees-peggy-collins.html](https://drive.google.com/uc?export=view&id=1mLWum5C1m0u9Tb5deDg8nVLnVv5cxZMM)

**Dataset:** Let's revisit the 'UniversalBank.csv' dataset that we looked at in the k-NN lab.

![picture](https://drive.google.com/uc?export=view&id=1oyxN7J-WBW5kk_Z1mz2-zG8Uue-fRaqC)

Universal Bank is a relatively young bank growing rapidly in terms of overall customer acquisition. The majority of these customers are liability customers (depositors) with varying sizes of relationship with the bank. The customer base of asset customers (borrowers) is quite small, and **the bank is interested in expanding this base rapidly to bring in more loan business.** In particular, it wants to explore ways of converting its depositors to personal loan customers (while retaining them as depositors).

A campaign that the bank ran last year for depositors showed a healthy conversion rate of over 9% success. This has encouraged the retail marketing department to devise smarter campaigns with better target marketing. **The goal is to replace
k-NN used last time, by a more sophisticated Decision Tree, to predict whether a new customer will accept a loan offer.** This will serve as the basis for the design of a new campaign.

The file _UniversalBank.csv_ contains data on 5000 customers. The data include customer demographic information (age, income, etc.), the customer’s relationship with the bank (mortgage, securities account, etc.), and the customer response to the last personal loan campaign (Personal Loan). Among these 5000 customers, only 480 (= 9.6%) accepted the personal loan that was offered to them in the earlier campaign.

Columns: Age (years), Experience (years of work experience), Income (in thousands of USD), Family (number of family members), CCAvg (average monthly credit card spending, in thousands of USD), Education (1 undergrad, 2 grad, 3 professional degrees), Mortgage (value of house mortgage, in thousands of USD; 0 if none), personal loan (0 or 1), securities account (0 or 1), CD account (0 or 1), Online (0 or 1) and CreditCard (0 or 1).

# 1. Classification Trees

In [ ]:
# Load the libraries used throughout this lab.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score

## Preprocessing

In [ ]:
# Load the UniversalBank data and drop the two ID-like columns that carry no information.
bank_df = pd.read_csv('https://raw.githubusercontent.com/AnalyticsArmory/data/main/UniversalBank.csv')
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

In [ ]:
# Preview the first 10 customers.
bank_df.head(10)

### Education stays as 1, 2, 3

In the k-NN lab we turned `Education` into dummy variables (one-hot encoding), because 1/2/3 are labels for categories, not quantities. Here, to stay consistent with the textbook, we leave it as a single number. That means the tree treats Education as **ordered**: it can split on "Education ≤ 1.5" (undergrad vs. everyone else) but never on "graduate vs. the rest". For this dataset it makes little difference, but notice it is the opposite choice from the k-NN lab.

In [ ]:
# Define X (inputs/predictors) and y (output/outcome)
X = bank_df.drop(columns=['Personal Loan'])
y = bank_df['Personal Loan']

In [ ]:
# Hold out 40% of customers as a test set the tree never sees during training.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1)

## Build the 'Full' Tree

In [ ]:
# Grow the "full" tree: no limits, so it keeps splitting until every leaf is pure.
full_clf = DecisionTreeClassifier(random_state=0)
full_clf.fit(X_train, y_train)

In [ ]:
# How big did the full tree get?
print('Number of nodes', full_clf.tree_.node_count)

In [ ]:
# Print the full tree as text: one line per split.
print(tree.export_text(full_clf, feature_names=list(X.columns)))

In [ ]:
# Draw the full tree; 'filled' colors each node by its majority class.
plt.figure(figsize=(15,10))   # a big canvas, otherwise the tree is unreadable
tree.plot_tree(full_clf, filled=True);

In [ ]:
# Predict loan acceptance for the training and test customers separately.
y_pred_train = full_clf.predict(X_train)
y_pred_test = full_clf.predict(X_test)

In [ ]:
# Accuracy of full tree on training and test sets
train_score = accuracy_score(y_train, y_pred_train)
test_score = accuracy_score(y_test, y_pred_test)

print('Accuracy score of full tree on training data =', round(train_score*100,1), '%')
print('Accuracy score of full tree on testing data =', round(test_score*100,1), '%')

In [ ]:
# Confusion matrices of the full tree on the training and test sets.
print("Confusion Matrix of full tree on train data:")
print(confusion_matrix(y_train, y_pred_train))

print("Confusion Matrix of full tree on test data:")
print(confusion_matrix(y_test, y_pred_test))

**Question 1.** Compare the full tree's accuracy and confusion matrix on the training data with those on the test data. What does the gap tell you about this tree?

**Answer**

*Provide your answer here*

## Build a 'Reduced' Tree (less prone to overfitting)

In [ ]:
# Grow a "reduced" tree by limiting how far it may split.
# All options: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
#   max_depth             : maximum number of levels from the root to a leaf
#   min_samples_split     : minimum number of customers a node needs before it may split
#   min_samples_leaf      : minimum number of customers allowed in every leaf
#   min_impurity_decrease : minimum drop in impurity needed to justify a split
#   criterion             : 'gini' (default) or 'entropy'
clf = DecisionTreeClassifier(
      max_depth=10,
      min_samples_split = 40,
      min_samples_leaf = 20,
      min_impurity_decrease = 0.01,
      criterion = 'gini')
clf.fit(X_train, y_train)

In [ ]:
# Plot the reduced tree with readable feature and class names.
plt.figure(figsize=(15,10))
cnames = ['no :(','yes! :)']
tree.plot_tree(clf, fontsize=12, feature_names=X.columns, class_names=cnames, filled=True);

In [ ]:
# Accuracy and confusion matrix of the reduced tree on the test set.
y_pred = clf.predict(X_test)

print('Accuracy score of reduced tree on test data =', round(accuracy_score(y_test, y_pred)*100,1), '%')

print("Confusion Matrix of reduced tree on test data:")
confusion_matrix(y_test, y_pred)

## Should we contact our newest customer, 'Isabel' with a loan offer?

In [ ]:
# Describe the new customer, Isabel, with the same columns as X.
isabel = pd.DataFrame([
{
'Age':31,
 'Experience':5,
 'Income':184,
 'Family':3,
 'CCAvg':2.9,
 'Education':3,
 'Mortgage':105,
 'Securities Account':0,
 'CD Account':0,
 'Online':0,
 'CreditCard':1
 }
 ])

In [ ]:
# Ask the reduced tree whether Isabel will accept the loan (1 = yes, 0 = no).
clf.predict(isabel)

**Question 2.** Follow Isabel's path down the reduced tree plotted earlier, and check it against the prediction above. Which splits does she pass through, what does the tree predict, and should the bank contact her with a loan offer?

**Answer**

*Provide your answer here*

# 2. Bonus Content for Classification Trees: K-fold, Grid Search, Random Forest

## Bonus: K-fold cross-validation

In [ ]:
# 5-fold cross-validation of the reduced tree on the training data.
accuracy_scores = cross_val_score(clf, X_train, y_train, cv=5)
print('Accuracy scores of each fold: ', [f'{acc:.3f}' for acc in accuracy_scores])
print('5-fold cross-validation mean accuracy:', round(accuracy_scores.mean(),3))

## Bonus: Grid search over parameters to optimize the tree

In [ ]:
# Try every combination of these settings, scoring each with 5-fold cross-validation.
param_grid = {
    'max_depth': [10, 20, 30, 40],
    'min_samples_leaf': [20, 50, 100, 200],
    'min_impurity_decrease': [0, 0.0005, 0.001, 0.005, 0.01],
}

gridSearch = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5, scoring = 'accuracy')
gridSearch.fit(X_train, y_train)

print('Best accuracy: ', gridSearch.best_score_)
print('Parameter values to achieve best accuracy: ', gridSearch.best_params_)

## Bonus: Random Forests

![picture](https://drive.google.com/uc?export=view&id=1ZzAstlbZbrKK6im9_vBJVmDo42Odmi12)



In [ ]:
# Load the random forest classifier.
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Grow a forest of random trees; each sees a random sample of customers and columns,
# so the forest overfits much less than a single full tree.
rf = RandomForestClassifier(
    n_estimators=500,   # number of trees in the forest
    random_state=0,
    criterion='gini',
    n_jobs = -1)          # use every CPU core
rf.fit(X_train, y_train)

In [ ]:
# Accuracy and confusion matrix of the random forest on the test set.
print("Accuracy=", accuracy_score(y_test, rf.predict(X_test)))
confusion_matrix(y_test, rf.predict(X_test))

Even though we cannot visualize Random Forests easily, we can still get a sense of the average importance of each feature in the dataset, using the forest's `feature_importances_` attribute. Feature importance is based on mean decrease in impurity (averaging across all the trees generated in the random forest)

In [ ]:
# Rank the features by how much they reduced impurity in the random forest (averaged over all its trees).
importance_df = pd.DataFrame({'feature': X_train.columns, 'importance': rf.feature_importances_})
importance_df = importance_df.sort_values('importance', ascending=False)

importance_df.plot(kind='bar', x='feature')
plt.tight_layout()
plt.show()

# 3. Regression Trees

Let's now reset, and focus on a different objective: trying to predict **'Income'** based on the other features of the dataset. The Decision Tree will work the same with one important difference: the criterion to evaluate accuracy will no longer be the gini score, but rather, the sum of squared errors (since we are trying to predict 'income' which is a numerical variable).

In [ ]:
# Load the regression tree, which predicts a number instead of a class.
from sklearn.tree import DecisionTreeRegressor

In [ ]:
# A reminder of what the data looks like.
bank_df.head()

In [ ]:
# Now Income is the outcome and every other column is a predictor.
y_reg = bank_df['Income']
X_reg = bank_df.drop(columns=['Income'])

In [ ]:
# Hold out 40% of customers as a test set.
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.4, random_state=0)

In [ ]:
# Grow a regression tree, kept small (max depth 5) so it stays readable.
reg_tree = DecisionTreeRegressor(
      max_depth=5,
      min_samples_split = 40,
      min_samples_leaf = 100,
      min_impurity_decrease = 0.01,
      criterion = 'squared_error')
reg_tree.fit(X_train_reg, y_train_reg)

In [ ]:
# Plot the regression tree; each leaf shows the average Income it predicts.
plt.figure(figsize=(25,10))
tree.plot_tree(reg_tree, fontsize=12, feature_names=X_reg.columns, filled=True);

In [ ]:
# Evaluate the regression tree on the test set: RMSE (in $000s) and R-squared.
y_pred_reg = reg_tree.predict(X_test_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)
print("RMSE=", round(rmse,3), "R2=", round(r2,3))